# Modelo A — integración Google Colab para A8.5
Notebook fino de orquestación. El entrenamiento queda desactivado hasta una decisión explícita del usuario.

In [ ]:
# ÚNICOS PARÁMETROS OPERATIVOS EDITABLES
REPO_URL = 'https://github.com/OWNER/REPOSITORY.git'
GIT_COMMIT = ''  # SHA completo (40 hex) de main después del merge; obligatorio
DRIVE_BASE = '/content/drive/MyDrive/model_a'
DRIVE_MUTANTS_HDF5 = DRIVE_BASE + '/data/proc_483p.hdf5'
DRIVE_WT_HDF5 = DRIVE_BASE + '/data/wt_companion.hdf5'
DRIVE_MODEL_A_RUNS_ROOT = DRIVE_BASE + '/runs/model_a_a8_5'
REQUESTED_DEVICE = 'auto'
EXECUTION_MODE = 'preflight'  # preflight | fresh | resume
RESUME_CHECKPOINT = ''  # last.pt explícito; nunca se descubre automáticamente
RUN_FRESH = False
RUN_RESUME = False
LOCAL_ROOT = '/content/model_a_workspace'
REPO_DIR = LOCAL_ROOT + '/repo'
STAGING_ROOT = LOCAL_ROOT + '/staging'
if EXECUTION_MODE not in {'preflight', 'fresh', 'resume'}: raise ValueError('EXECUTION_MODE inválido')
if RUN_FRESH != (EXECUTION_MODE == 'fresh'): raise ValueError('RUN_FRESH solo puede activarse en modo fresh')
if RUN_RESUME != (EXECUTION_MODE == 'resume'): raise ValueError('RUN_RESUME solo puede activarse en modo resume')
if RUN_FRESH and RUN_RESUME: raise ValueError('Fresh y resume son mutuamente excluyentes')


## 1. Runtime Colab, GPU y montaje de Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import re, subprocess, sys
drive.mount('/content/drive')
if not re.fullmatch(r'[0-9a-fA-F]{40}', GIT_COMMIT): raise ValueError('GIT_COMMIT debe ser un SHA completo de 40 hexadecimales')
for label, raw in [('mutants', DRIVE_MUTANTS_HDF5), ('WT companion', DRIVE_WT_HDF5)]:
    path = Path(raw).resolve()
    if not path.is_file() or path.stat().st_size <= 0: raise FileNotFoundError(f'HDF5 {label} ausente o vacío: {path}')


## 2. Clon y checkout reproducible de un SHA explícito

In [ ]:
repo = Path(REPO_DIR).resolve(); local_root = Path(LOCAL_ROOT).resolve()
if repo == local_root or not repo.is_relative_to(local_root): raise ValueError('REPO_DIR fuera del workspace')
local_root.mkdir(parents=True, exist_ok=True)
fresh_clone = not (repo / '.git').is_dir()
if fresh_clone:
    if repo.exists(): raise RuntimeError('REPO_DIR existe pero no es un clon Git')
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(repo)], check=True)
elif subprocess.run(['git','-C',str(repo),'status','--porcelain'], check=True, capture_output=True, text=True).stdout.strip():
    raise RuntimeError('El clon controlado está sucio')
subprocess.run(['git','-C',str(repo),'fetch','--tags','--prune','origin'], check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',GIT_COMMIT], check=True)
HEAD = subprocess.run(['git','-C',str(repo),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if HEAD != GIT_COMMIT.lower(): raise RuntimeError(f'HEAD {HEAD} no coincide con SHA solicitado {GIT_COMMIT}')
sys.path[:0] = [str(repo), str(repo/'src')]
from scripts.colab_preflight import git_revision
git_revision(repo, GIT_COMMIT.lower())


## 3. Requirements de Colab y diagnóstico Python / PyTorch / PyG / CUDA

In [ ]:
import platform, torch
from scripts.colab_preflight import prepare_colab_environment
ENVIRONMENT = prepare_colab_environment(repo_root=repo, marker_root=local_root, commit=HEAD, requirements_path=repo/'requirements-colab.txt', device=REQUESTED_DEVICE, python_version=platform.python_version(), torch_version=torch.__version__)
RUNTIME = ENVIRONMENT['runtime']; print(RUNTIME)


## 4. Staging local seguro, tamaño, SHA256 y solo lectura

In [ ]:
from scripts.colab_preflight import stage_file, require_free_space
staging = Path(STAGING_ROOT).resolve(); staging.mkdir(parents=True, exist_ok=True)
required = Path(DRIVE_MUTANTS_HDF5).stat().st_size + Path(DRIVE_WT_HDF5).stat().st_size
require_free_space(staging, required * 2)
MUTANTS_RECORD = stage_file(DRIVE_MUTANTS_HDF5, staging/'proc_483p.hdf5', staging_root=staging, role='mutants')
WT_RECORD = stage_file(DRIVE_WT_HDF5, staging/'wt_companion.hdf5', staging_root=staging, role='wt_companion')
for record in (MUTANTS_RECORD, WT_RECORD): Path(record['local_locator']).chmod(0o444)
print(MUTANTS_RECORD); print(WT_RECORD)


## 5. Configuración resuelta desde model_a_pilot.yaml y split congelado

In [ ]:
import os
from scripts.colab_preflight import confined_path, generate_runtime_config, preflight_output_root
os.chdir(repo)
drive_base = Path(DRIVE_BASE).resolve()
OUTPUT_ROOT = confined_path(DRIVE_MODEL_A_RUNS_ROOT, drive_base, label='Model A run_root')
if any(part.lower().startswith('model_b') for part in OUTPUT_ROOT.parts): raise ValueError('run_root A no puede pertenecer al namespace B')
preflight_output_root(OUTPUT_ROOT, allowed_root=drive_base)
RESOLVED_CONFIG = local_root/'resolved_configs'/'model_a_pilot_runtime.yaml'
runtime_overrides = {
 'paths.mutants_hdf5': MUTANTS_RECORD['local_locator'],
 'paths.wt_companion_hdf5': WT_RECORD['local_locator'],
 'outputs.root_dir': str(OUTPUT_ROOT),
 'training.device': REQUESTED_DEVICE}
if EXECUTION_MODE == 'resume': runtime_overrides['training.epochs'] = 6  # una época adicional mínima
CONFIG_RESULT = generate_runtime_config(repo/'configs/model_a_pilot.yaml', RESOLVED_CONFIG, overrides=runtime_overrides)
assert CONFIG_RESULT['config']['split']['persist_path'] == 'splits/leave_position_out_seed_42.json'
assert CONFIG_RESULT['config']['split']['allow_create'] is False


## 6. Preflight científico específico del Modelo A y resumen previo

In [ ]:
import json
from scripts.colab_preflight import validate_model_a_preflight
PREFLIGHT_MODE = 'resume' if EXECUTION_MODE == 'resume' else 'fresh'
PREFLIGHT = validate_model_a_preflight(RESOLVED_CONFIG, repo_root=repo, expected_commit=HEAD, mode=PREFLIGHT_MODE, resume_from=(RESUME_CHECKPOINT or None) if PREFLIGHT_MODE == 'resume' else None)
PREFLIGHT['runtime'] = RUNTIME
print(json.dumps(PREFLIGHT, indent=2, default=str))


## 7. FRESH RUN A8.5 — preparado, nunca automático

In [ ]:
from scripts.colab_preflight import build_train_command, run_command
FRESH_COMMAND = build_train_command(repo, RESOLVED_CONFIG, device=REQUESTED_DEVICE)
print('FRESH RUN command:', FRESH_COMMAND)
if RUN_FRESH:
    FRESH_RESULT = run_command(FRESH_COMMAND, cwd=repo)
    print(FRESH_RESULT.stdout)
    if FRESH_RESULT.returncode: raise subprocess.CalledProcessError(FRESH_RESULT.returncode, FRESH_COMMAND, FRESH_RESULT.stdout, FRESH_RESULT.stderr)


## 8. RESUME explícito desde last.pt — preparado, no ejecutado automáticamente

In [ ]:
RESUME_COMMAND = None
if EXECUTION_MODE == 'resume':
    checkpoint = Path(RESUME_CHECKPOINT).resolve()
    RESUME_COMMAND = build_train_command(repo, RESOLVED_CONFIG, device=REQUESTED_DEVICE, resume_from=checkpoint)
    print('RESUME command:', RESUME_COMMAND)
    if RUN_RESUME:
        RESUME_RESULT = run_command(RESUME_COMMAND, cwd=repo)
        print(RESUME_RESULT.stdout)
        if RESUME_RESULT.returncode: raise subprocess.CalledProcessError(RESUME_RESULT.returncode, RESUME_COMMAND, RESUME_RESULT.stdout, RESUME_RESULT.stderr)


## 9. Ubicación de artefactos
El trainer versionado crea bajo `run_root/model_a_nodal_multiscale_pair/run_<id>/`: `run_manifest.json`, `config_resolved.yaml`, `metrics.jsonl`, auditoría de gradientes y `checkpoints/{best,last}.pt`. La copia del split dentro del run es un artefacto byte-identical creado por la infraestructura común; la fuente sigue siendo el split congelado del repositorio.